# Hands-on 4 — Few-shot land cover: labels are cheap now

**Chapter 4 · Working with embeddings**

The classic land-cover workflow is: collect thousands of training samples, engineer features (indices, textures, temporal statistics), train a random forest, repeat for every new region and year. Embeddings change the economics: the features are already there, so a **simple linear classifier on a few dozen labelled pixels** goes a long way. In this notebook you will:

1. get "free" labels from ESA WorldCover and understand their limits,
2. train a logistic regression on AlphaEarth, on TESSERA and on raw Sentinel-2 bands, with 5 to 500 labels per class, and compare learning curves,
3. read a confusion matrix and see which classes embeddings confuse and why,
4. map the whole area, and
5. do it **without any labels** with k-means clustering.

Requires the caches from Hands-on 1 and 2. Run time about 4 minutes, no new downloads.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from affine import Affine
from rasterio.crs import CRS
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

import eoemb

AOI = (23.90, 38.00, 24.00, 38.10)
cache = eoemb.DATA_DIR / "cache"

z = np.load(cache / "aef_attica_2024.npz")
transform = Affine.from_gdal(*z["transform"]); crs = CRS.from_string(str(z["crs"]))
aef = eoemb.dequantize(z["raw"]); H, W = aef.shape[1:]
extent = eoemb.lonlat_extent(transform, crs, (H, W))

t0 = time.time()
tes = eoemb.tessera_read_like(AOI, 2024, transform, crs, (H, W), embeddings_dir=cache / "tessera")
print(f"TESSERA from local tiles in {time.time() - t0:.0f} s")

with rasterio.open(eoemb.DATA_DIR / "s2_attica_20240713_l2a.tif") as src:
    s2 = src.read().astype(np.float32) / 10_000          # red, green, blue, nir reflectance
with rasterio.open(eoemb.DATA_DIR / "s2_attica_20240713_rgb.tif") as src:
    s2_rgb = np.moveaxis(src.read(), 0, -1)
ndvi = (s2[3] - s2[0]) / (s2[3] + s2[0] + 1e-6)

## 1. Labels from ESA WorldCover

WorldCover is a global 10 m land-cover map (2021, version 2) produced by ESA. It is itself a machine-learning product with about 75 % overall accuracy, and it is three years older than our embeddings. We use it as a *cheap source of labels*, the way you might use an old cadastre, OpenStreetMap or a national map: convenient, plentiful and **noisy**. Keep that in mind when you read accuracies below: part of the "error" is the reference being wrong.

We keep five classes that exist in the area.

In [ ]:
with rasterio.open(eoemb.DATA_DIR / "worldcover_2021_attica.tif") as src:
    wc = src.read(1)

CLASSES = {10: "tree cover", 20: "shrubland", 30: "grassland", 50: "built-up", 80: "water"}
COLORS = {10: "#006400", 20: "#ffbb22", 30: "#ffff4c", 50: "#fa0000", 80: "#0064c8"}

counts = pd.Series({name: int((wc == k).sum()) for k, name in CLASSES.items()})
print((counts / 100).round(0).astype(int).rename("hectares"))

from matplotlib.colors import ListedColormap, BoundaryNorm
cmap = ListedColormap([COLORS[k] for k in CLASSES]); norm = BoundaryNorm(list(CLASSES) + [200], cmap.N)
fig, ax = plt.subplots(1, 2, figsize=(14, 6.5))
ax[0].imshow(s2_rgb, extent=extent); ax[0].set_title("Sentinel-2, July 2024")
wc_show = np.where(np.isin(wc, list(CLASSES)), wc, np.nan)
ax[1].imshow(wc_show, extent=extent, cmap=cmap, norm=norm); ax[1].set_title("ESA WorldCover 2021")
plt.tight_layout()

Look at the shrubland/grassland boundary on the mountain and compare with the image: in a Mediterranean landscape these classes are a continuum, and the map draws a hard line through it. Expect confusion there from any classifier, and do not blame the embedding for it.

## 2. Setting up a fair experiment

Three feature sets for the same pixels:

| features | dimensions | what it represents |
|---|---|---|
| AlphaEarth 2024 | 64 | a year of multi-sensor data, learned |
| TESSERA 2024 | 128 | a year of Sentinel-1 + 2, learned |
| Sentinel-2 bands + NDVI, one date | 5 | the classic starting point: red, green, blue, NIR, NDVI |

One classifier for all: **logistic regression** (a linear model) after standardising each feature. If embeddings are as good as claimed, a linear model should be enough, because the hard, non-linear work has been done by the foundation model.

**Spatial split.** Pixels next to each other are nearly identical, so a random train/test split would leak information and inflate accuracy. We use a 1 km checkerboard: train from black squares, test from white squares.

In [ ]:
features = {
    "AlphaEarth (64)": aef.reshape(64, -1).T,
    "TESSERA (128)": tes.reshape(128, -1).T,
    "S2 bands + NDVI (5)": np.vstack([s2, ndvi[None]]).reshape(5, -1).T,
}
y = wc.ravel()
usable = np.isin(y, list(CLASSES)) & ~np.isnan(aef[0].ravel()) & ~np.isnan(tes[0].ravel())

rows, cols = np.divmod(np.arange(H * W), W)
checker = ((rows // 100) + (cols // 100)) % 2            # 100 px = 1 km
train_pool = usable & (checker == 0)
test_pool = usable & (checker == 1)

rng = np.random.default_rng(0)
test_idx = np.concatenate([rng.choice(np.flatnonzero(test_pool & (y == k)), 1000, replace=False) for k in CLASSES])
print(f"test set: {len(test_idx)} pixels, 1000 per class, from the white squares")

plt.figure(figsize=(5, 5)); plt.imshow(checker.reshape(H, W), extent=extent, cmap="gray"); plt.title("1 km checkerboard: train (black) / test (white)");

## 3. Learning curves: how many labels do you need?

In [ ]:
def fit_eval(X, train_idx):
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
    clf.fit(X[train_idx], y[train_idx])
    pred = clf.predict(X[test_idx])
    return clf, accuracy_score(y[test_idx], pred), f1_score(y[test_idx], pred, average="macro")

N_PER_CLASS = [5, 10, 20, 50, 100, 200, 500]
REPEATS = 5
results = []
t0 = time.time()
for n in N_PER_CLASS:
    for rep in range(REPEATS):
        r = np.random.default_rng(100 * n + rep)
        train_idx = np.concatenate([r.choice(np.flatnonzero(train_pool & (y == k)), n, replace=False) for k in CLASSES])
        for name, X in features.items():
            _, acc, f1 = fit_eval(X, train_idx)
            results.append({"features": name, "n_per_class": n, "repeat": rep, "accuracy": acc, "macro_f1": f1})
print(f"{len(results)} models trained in {time.time() - t0:.0f} s")
curve = pd.DataFrame(results).groupby(["features", "n_per_class"]).agg(acc_mean=("accuracy", "mean"), acc_std=("accuracy", "std")).reset_index()

plt.figure(figsize=(8, 5))
for name in features:
    d = curve[curve.features == name]
    plt.errorbar(d.n_per_class, d.acc_mean, yerr=d.acc_std, marker="o", capsize=3, label=name)
plt.xscale("log"); plt.xlabel("labelled pixels per class"); plt.ylabel("accuracy on 5 000 test pixels")
plt.title("Learning curves, logistic regression, 5 WorldCover classes"); plt.grid(alpha=.3); plt.legend();
curve.pivot(index="n_per_class", columns="features", values="acc_mean").round(3)

Three things to take from the curves:

- **With 5 labels per class** (25 pixels in total!) both embeddings already beat the spectral baseline, and the gap does not close as labels are added: single-date bands simply do not contain the information.
- **The embeddings saturate**: from a few hundred labels per class onwards, more labels barely help. What limits accuracy is the reference map and the fuzzy class definitions, not the model.
- **AlphaEarth and TESSERA are close**. Which one wins depends on the classes, the region, the year, and sometimes the random draw. Test on your own task before you commit.

Absolute numbers (~75 %) look modest. Remember that 25–30 % of the WorldCover pixels we test against are probably wrong or ambiguous themselves (shrub vs grass, single trees vs tree cover), so 75 % agreement with WorldCover is not 75 % correct.

## 4. Where are the errors? Confusion matrix at 100 labels per class

In [ ]:
r = np.random.default_rng(1)
train_idx = np.concatenate([r.choice(np.flatnonzero(train_pool & (y == k)), 100, replace=False) for k in CLASSES])
fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
models = {}
for ax, (name, X) in zip(axes, features.items()):
    clf, acc, f1 = fit_eval(X, train_idx); models[name] = clf
    cm = confusion_matrix(y[test_idx], clf.predict(X[test_idx]), labels=list(CLASSES), normalize="true")
    ConfusionMatrixDisplay(cm, display_labels=list(CLASSES.values())).plot(ax=ax, colorbar=False, values_format=".2f", xticks_rotation=45)
    ax.set_title(f"{name}\naccuracy {acc:.2f}")
plt.tight_layout()

Water and built-up are easy for everyone. The three vegetation classes are where the differences are: raw bands confuse tree cover with shrubland heavily (in July both are just "dark green"), embeddings separate trees much better because they have seen the whole year. Shrubland vs grassland remains hard for all, and that is the fuzzy boundary we saw in the reference map.

## 5. Map the whole area

Predicting a million pixels with a linear model takes a second.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 6.5))
for ax, (name, X) in zip(axes, features.items()):
    pred = np.full(H * W, np.nan)
    pred[usable] = models[name].predict(X[usable])
    ax.imshow(pred.reshape(H, W), extent=extent, cmap=cmap, norm=norm); ax.set_title(f"{name}, 100 labels/class")
plt.tight_layout()

Compare with the WorldCover map in section 1. The embedding maps are cleaner, with less salt-and-pepper, because each embedding already summarises a neighbourhood and a year. Note also that both embedding maps show 2024 and disagree with the 2021 reference where things changed (new buildings, the August 2024 Penteli fire on the west side).

## 6. No labels at all: k-means

If you have no labels, cluster. k-means groups pixels whose embeddings are close; you then name the clusters by looking at them. This is exactly the unsupervised classification you know from remote sensing software, but in an embedding space where "close" means "similar kind of place", not "similar reflectance".

In [ ]:
from sklearn.cluster import KMeans

K = 8
X = features["AlphaEarth (64)"]
sub = rng.choice(np.flatnonzero(usable), 50_000, replace=False)
km = KMeans(n_clusters=K, n_init=4, random_state=0).fit(X[sub])
clusters = np.full(H * W, -1); clusters[usable] = km.predict(X[usable])

fig, ax = plt.subplots(1, 2, figsize=(14, 6.5))
ax[0].imshow(s2_rgb, extent=extent); ax[0].set_title("Sentinel-2")
ax[1].imshow(np.where(clusters >= 0, clusters, np.nan).reshape(H, W), extent=extent, cmap="tab10", vmin=0, vmax=9); ax[1].set_title(f"k-means, k = {K}, on AlphaEarth")
plt.tight_layout()

# how do clusters relate to WorldCover?
ct = pd.crosstab(pd.Series(clusters[sub], name="cluster"), pd.Series([CLASSES[k] for k in y[sub]], name="WorldCover"))
(ct.div(ct.sum(axis=1), axis=0) * 100).round(0).astype(int)

Each row is a cluster; the columns say what WorldCover calls those pixels. Some clusters are pure (water, built-up, tree cover), others split shrubland into several sub-types that WorldCover lumps together, which may be exactly the detail you want. Choosing *k* is a judgement call: more clusters, finer distinctions, more naming work.

---
## Exercises

**E1.** Replace logistic regression with a random forest (`RandomForestClassifier(n_estimators=200)`) and re-run the learning curve for AlphaEarth only. Does a non-linear model help? What does that tell you about the embedding space?

**E2.** Concatenate AlphaEarth and TESSERA into one 192-dimensional feature vector. Is the combination better than either alone at 100 labels per class?

**E3.** Label your own pixels: choose 10 locations per class on the Sentinel-2 image (`plt.imshow(s2_rgb, extent=extent)` and read coordinates off the axes), train on them, predict the map. Compare with the WorldCover-trained map. Where do you trust your labels more than WorldCover's?